# Schuhhersteller: Verkaufsprognose für Badeschlappen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_09/schuhhersteller_simple.ipynb)

**Lernziel:** Lineare und polynomiale Regression anhand von R² vergleichen, das Modell auf einem späteren Validierungsjahr auswählen und auf einem unberührten zukünftigen Testjahr bewerten.

**Szenario:** Vorhersage der Badeschlappen-Verkäufe basierend auf Temperatur für verschiedene Sommer-Szenarien

**Lehrbeispiel:** Die Daten sind simuliert und basieren nicht auf realen Zusammenhängen. Sie stehen in keinem Bezug zu einem realen Unternehmen. :-)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.base import clone

# Historische Daten der letzten 3 Jahre (156 Wochen)
np.random.seed(42)
n_wochen = 156  # 3 Jahre

# Vereinfachte saisonale Temperatur; Woche 1 ist keine feste Kalenderwoche.
wochen = np.arange(1, n_wochen + 1)
# Grundtemperatur mit saisonalem Muster
temperatur = 12 + 15 * np.sin(2 * np.pi * wochen / 52) + np.random.normal(0, 3, n_wochen)

# Badeschlappen-Verkäufe: stark temperaturabhängig
# Niedriger Grundverkauf, überproportionaler Anstieg oberhalb der Schwelle.
basis_verkaeufe = 50  # Grundverkauf auch im Winter
# Starker Temperatur-Effekt für Badeschlappen
temp_schwelle = 15  # Ab 15°C wird's interessant
# Sicherstellen, dass nur positive Werte für Potenz verwendet werden
temp_diff = np.maximum(temperatur - temp_schwelle, 0)  # Negative Werte auf 0 setzen
temp_effekt = temp_diff ** 1.5 * 40  # Potenzfunktion, keine Exponentialfunktion

rauschen = np.random.normal(0, 30, n_wochen)
verkaeufe = basis_verkaeufe + temp_effekt + rauschen
verkaeufe = np.maximum(verkaeufe, 10)  # Mindestens 10 Paar pro Woche

df = pd.DataFrame({
    'woche': wochen,
    'temperatur': temperatur.round(1),
    'verkaeufe': verkaeufe.round(0).astype(int)
})

print(f"Historische Daten: {n_wochen} Wochen (3 Jahre)")
print("\nErste 10 Wochen:")
df.head(10)

## 1. Vergangenheit zum Lernen, Zukunft zum Prüfen

Wir behalten die zentrale Frage: **Passt eine Gerade oder eine quadratische Kurve besser?** Ein höheres Trainings-R² reicht als Antwort nicht aus.

| Zeitraum | Rolle | Verwendung |
|---|---|---|
| Jahr 1, Wochen 1–52 | Training | Beide vorab festgelegten Modelle fitten |
| Jahr 2, Wochen 53–104 | Validierung | Modell mit dem höheren Validierungs-R² wählen |
| Jahr 3, Wochen 105–156 | Test | Gewähltes Modell nach Refit auf Jahren 1–2 einmal bewerten |

Kein zufälliges Mischen: In einer Prognoseaufgabe darf die Zukunft nicht beim Lernen helfen. Jedes Intervall enthält einen vollständigen saisonalen Zyklus. Auch die erste Datenexploration beschränkt sich auf Jahr 1.

**R²:** 1 bedeutet perfekte Vorhersagen; 0 entspricht dem konstanten Mittelwert der jeweils bewerteten Zielwerte; negative Werte sind möglich. Zusätzlich berichten wir den **MAE in Paar pro Woche**, damit die Fehlergröße greifbar bleibt. Beide Modelle werden auf denselben Validierungszeilen verglichen.

Ein einzelnes Validierungsjahr ist eine einfache Lehrlösung, keine Garantie für jedes zukünftige Jahr. Bei mehr Daten wären mehrere zeitlich vorwärts laufende Validierungsfenster sinnvoll.

In [ ]:
train_df = df.iloc[:52].copy()
val_df = df.iloc[52:104].copy()
test_df = df.iloc[104:].copy()

X_train, y_train = train_df[['temperatur']], train_df['verkaeufe']
X_val, y_val = val_df[['temperatur']], val_df['verkaeufe']
X_test, y_test = test_df[['temperatur']], test_df['verkaeufe']

assert train_df['woche'].max() < val_df['woche'].min()
assert val_df['woche'].max() < test_df['woche'].min()
assert len(train_df) == len(val_df) == len(test_df) == 52
for name, interval in [('Training', train_df), ('Validierung', val_df), ('Test', test_df)]:
    print(f"{name}: Wochen {interval['woche'].min()}–{interval['woche'].max()} ({len(interval)} Zeilen)")

In [ ]:
# Exploration nur auf dem Trainingsjahr.
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(train_df['woche'], train_df['temperatur'], 'r-', label='Temperatur', linewidth=2)
ax1_twin = ax1.twinx()
ax1_twin.plot(train_df['woche'], train_df['verkaeufe'], 'b-', label='Badeschlappen-Verkäufe', linewidth=2)
ax1.set_xlabel('Woche')
ax1.set_ylabel('Temperatur (°C)', color='red')
ax1_twin.set_ylabel('Verkäufe (Paar)', color='blue')
ax1.set_title('Saisonaler Verlauf: nur Trainingsjahr 1')

# Streudiagramm: Temperatur vs. Verkäufe
ax2.scatter(train_df['temperatur'], train_df['verkaeufe'], alpha=0.6, color='orange', s=20)
ax2.set_xlabel('Temperatur (°C)')
ax2.set_ylabel('Verkäufe (Paar)')
ax2.set_title('Zusammenhang: nur Trainingsjahr 1')

plt.tight_layout()
plt.show()

print(f"Korrelation im Trainingsjahr: {train_df['temperatur'].corr(train_df['verkaeufe']):.3f}")
print("Der Zusammenhang ist sichtbar nichtlinear. Hilft ein quadratisches Modell auf späteren Daten?")

## 2. Gerade gegen quadratische Kurve: Auswahl auf Jahr 2

Wir legen die Kandidaten **vor** der Auswertung fest: Grad 1 und Grad 2. Polynomiale Features bleiben linear in den Modellkoeffizienten, erlauben aber eine gekrümmte Temperatur-Verkaufs-Kurve.

Die Pipeline hält Feature-Erzeugung und Regression zusammen. Beide Kandidaten sehen beim Fit nur Jahr 1. Wir wählen anhand des R² auf Jahr 2; MAE und negative Vorhersagen sind ergänzende Diagnosen. Jahr 3 bleibt geschlossen.

In [ ]:
candidates = {
    'Linear (Grad 1)': make_pipeline(PolynomialFeatures(degree=1, include_bias=False), LinearRegression()),
    'Polynomial (Grad 2)': make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
}

rows = []
for name, candidate in candidates.items():
    candidate.fit(X_train, y_train)
    train_prediction = candidate.predict(X_train)
    val_prediction = candidate.predict(X_val)
    rows.append({
        'Modell': name,
        'R2_Training': r2_score(y_train, train_prediction),
        'R2_Validierung': r2_score(y_val, val_prediction),
        'MAE_Validierung': mean_absolute_error(y_val, val_prediction),
        'Negative_Prognosen_Validierung': int((val_prediction < 0).sum())
    })

validation_results = pd.DataFrame(rows).set_index('Modell')
print(validation_results.round(3).to_string())
best_name = validation_results['R2_Validierung'].idxmax()
print(f'\nGewählt nach Validierungs-R²: {best_name}')

temp_grid = pd.DataFrame({'temperatur': np.linspace(X_train['temperatur'].min(), X_train['temperatur'].max(), 200)})
plt.figure(figsize=(12, 6))
plt.scatter(X_train['temperatur'], y_train, color='gray', alpha=0.4, label='Training: Jahr 1')
plt.scatter(X_val['temperatur'], y_val, color='orange', alpha=0.5, label='Validierung: Jahr 2')
for name, candidate in candidates.items():
    score = validation_results.loc[name, 'R2_Validierung']
    plt.plot(temp_grid['temperatur'], candidate.predict(temp_grid), label=f'{name}, Validierungs-R²={score:.3f}')
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Temperatur (°C)')
plt.ylabel('Verkäufe (Paar/Woche)')
plt.title('Modellwahl ohne Testdaten: beide Modelle nur auf Jahr 1 trainiert')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Refit und einmaliger Zukunftstest auf Jahr 3

Die Modellwahl ist abgeschlossen. Wir fitten eine frische Kopie des gewählten Modells auf **Jahren 1 und 2 zusammen** und bewerten nur dieses Modell auf Jahr 3. Damit werden die Validierungsdaten für den finalen Fit nutzbar, ohne die Testdaten zum Lernen oder zur Gradwahl zu verwenden.

**Wichtig:** Das Testresultat ist eine abschließende Bewertung, keine weitere Auswahlrunde. Falls wir danach aufgrund der Testfehler Modell oder Einstellungen ändern, benötigen wir neue, unberührte Daten für eine unabhängige Bewertung.

**Plausibilität gehört dazu:** Verkäufe können nicht negativ sein. Wir zeigen die unveränderten Vorhersagen und zählen negative Werte, statt sie still auf null zu setzen. Ein gutes globales R² kann unplausible Winterwerte oder systematische Fehler in einem Temperaturbereich verdecken.

In [ ]:
development_df = pd.concat([train_df, val_df])
X_development = development_df[['temperatur']]
y_development = development_df['verkaeufe']
final_model = clone(candidates[best_name])
final_model.fit(X_development, y_development)
test_prediction = final_model.predict(X_test)

test_r2 = r2_score(y_test, test_prediction)
test_mae = mean_absolute_error(y_test, test_prediction)
negative_test_predictions = int((test_prediction < 0).sum())
print(f'Gewähltes Modell: {best_name}; Fit auf 104 Wochen, Test auf 52 späteren Wochen')
print(f'Test-R²: {test_r2:.3f}; Test-MAE: {test_mae:.1f} Paar/Woche')
print(f'Negative Test-Prognosen: {negative_test_predictions}; Minimum: {test_prediction.min():.1f} Paar')
if negative_test_predictions:
    print('WARNUNG: Trotz gutem R² sind einige Prognosen physikalisch unmöglich. Nicht ungeprüft zur Bestandsplanung verwenden.')
outside_range = ~X_test['temperatur'].between(X_development['temperatur'].min(), X_development['temperatur'].max())
print(f'Testtemperaturen außerhalb des Fit-Bereichs (Extrapolation): {outside_range.sum()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(test_df['woche'], y_test, label='Beobachtung: Jahr 3')
axes[0].plot(test_df['woche'], test_prediction, label='Prognose: Fit auf Jahren 1–2')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set(xlabel='Woche', ylabel='Verkäufe (Paar/Woche)', title='Unberührtes Zukunftsjahr')
axes[0].legend()
axes[1].scatter(X_test['temperatur'], y_test - test_prediction, alpha=0.7)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set(xlabel='Temperatur (°C)', ylabel='Ist minus Prognose (Paar/Woche)', title='Testfehler nach Temperatur')
plt.tight_layout()
plt.show()

## 4. Anwendung: bedingte Sommer-Szenarien

Nach Abschluss der Bewertung dürfen wir für eine Anwendung alle drei historischen Jahre zum Fit verwenden. **Modelltyp und Grad bleiben unverändert.** Dieser neue Anwendungsfit hat kein eigenes unabhängiges Testresultat; das oben berichtete Test-R² gehört zum Fit auf Jahren 1–2.

Die Temperaturen sind **angenommene Szenarien**, keine verlässliche Zehn-Wochen-Wettervorhersage. Wir berechnen: *Welche Verkäufe würde unser Modell bei diesen Temperaturen erwarten?* Wetterunsicherheit und Modellunsicherheit fehlen in diesen Punktprognosen.

Weil die Daten simuliert sind, können wir die Summen zusätzlich mit der **rauschfreien Erzeugungsformel** vergleichen. Diese Lehrkontrolle erfolgt erst nach der Modellwahl und ist bei echten Daten nicht verfügbar. Sie zeigt, warum selbst ein hohes globales R² keine genaue Bestandsplanung garantiert.

In [ ]:
deployment_model = clone(final_model)
deployment_model.fit(df[['temperatur']], df['verkaeufe'])

heisser_sommer = [25, 28, 32, 30, 27, 29, 31, 28, 26, 24]
kuehler_sommer = [18, 21, 23, 19, 22, 20, 24, 21, 19, 17]
scenarios = {'Heißer Sommer': heisser_sommer, 'Kühler Sommer': kuehler_sommer}
scenario_predictions = {}
scenario_rows = []
for name, temperatures in scenarios.items():
    scenario_X = pd.DataFrame({'temperatur': temperatures})
    prediction = deployment_model.predict(scenario_X)
    scenario_predictions[name] = prediction
    if (prediction < 0).any():
        print(f'WARNUNG: Negative Verkaufsprognosen im Szenario {name}; Modell nicht ungeprüft einsetzen.')
    if not scenario_X['temperatur'].between(df['temperatur'].min(), df['temperatur'].max()).all():
        print(f'WARNUNG: Szenario {name} enthält Temperaturen außerhalb des historischen Bereichs (Extrapolation).')
    reference = 50 + 40 * np.maximum(np.array(temperatures) - 15, 0) ** 1.5
    scenario_rows.append({
        'Szenario': name,
        'Modell_Summe_Paar': prediction.sum(),
        'Rauschfreie_Simulation_Paar': reference.sum(),
        'Abweichung_Prozent': 100 * (prediction.sum() / reference.sum() - 1)
    })
    print(f'\n{name}:')
    print(pd.DataFrame({'Woche': np.arange(1, 11), 'Temperatur': temperatures, 'Prognose_Paar': prediction}).round(1).to_string(index=False))

scenario_results = pd.DataFrame(scenario_rows).set_index('Szenario')
print('\nSummen über zehn Wochen; Simulationsvergleich nur als Lehrkontrolle:')
print(scenario_results.round(1).to_string())
difference = scenario_results.loc['Heißer Sommer', 'Modell_Summe_Paar'] - scenario_results.loc['Kühler Sommer', 'Modell_Summe_Paar']
print(f'Bedingter Mengenunterschied laut Modell: {difference:.0f} Paar; keine Umsatzprognose ohne Preise.')

scenario_grid = pd.DataFrame({'temperatur': np.linspace(df['temperatur'].min(), df['temperatur'].max(), 200)})
plt.figure(figsize=(12, 6))
plt.scatter(df['temperatur'], df['verkaeufe'], alpha=0.3, color='gray', label='Alle historischen Daten (nach Testabschluss)')
plt.plot(scenario_grid['temperatur'], deployment_model.predict(scenario_grid), color='green', label=f'{best_name}: Anwendungsfit auf Jahren 1–3')
for name, temperatures in scenarios.items():
    plt.scatter(temperatures, scenario_predictions[name], s=70, label=name)
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Temperatur (°C)')
plt.ylabel('Verkäufe (Paar/Woche)')
plt.title('Bedingte Szenarien, keine garantierten Verkaufsprognosen')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Modell-Vergleich & Erkenntnisse

- **R² bleibt die zentrale Vergleichsgröße**, aber für die Modellwahl zählt das spätere Validierungsjahr, nicht der Trainingsfit. MAE ergänzt die Interpretation in Paar pro Woche.
- Das quadratische Modell kann den nichtlinearen Zusammenhang besser annähern. Der simulierte Mechanismus ist eine **Potenzfunktion oberhalb von 15°C**, kein exponentieller Anstieg und auch kein globales quadratisches Gesetz.
- **Auswahl → Refit → Zukunftstest:** Grad auf Jahr 2 wählen, gewähltes Modell auf Jahren 1–2 refitten, einmal auf Jahr 3 bewerten. Testdaten nicht zur nachträglichen Optimierung verwenden.
- Ein hohes Test-R² gilt für dieses Testjahr und diese Fehlergewichtung, nicht automatisch für jede Jahreszeit oder jede Bestandsentscheidung. Negative Prognosen und systematische Restfehler bleiben ernsthafte Grenzen beider Modellklassen.
- Der Anwendungsfit auf allen drei Jahren folgt erst nach der Bewertung und übernimmt den festgelegten Grad. Sein eigenes unabhängiges Testergebnis steht noch aus.
- Szenarien helfen, mögliche Mengenunterschiede zu diskutieren. Sie berücksichtigen hier weder Wetterunsicherheit noch Unsicherheitsintervalle, Preise, Lagerkosten oder entgangene Verkäufe.

**Diskussion / nächster Schritt:** Wie könnte ein Modell mit nichtnegativen Vorhersagen und einer plausiblen Temperaturschwelle aussehen? Eine neue Modellklasse oder eine Begrenzung auf null müsste erneut auf Trainings-/Validierungsdaten entwickelt werden; das bereits betrachtete Testjahr wäre dann kein unabhängiger Abschlusstest mehr.

**Vorsicht vor Overfitting:** Höhere Polynomgrade verbessern häufig das Trainings-R², können aber auf späteren Daten schlechter sein. Auch wiederholtes Ausprobieren auf demselben Validierungsjahr kann die Auswahl überanpassen.